# RL4EVRP Framework Demo
## Electric Vehicle Routing Problem with Deep Reinforcement Learning

This notebook demonstrates the refactored **rl4evrp** framework using:
- YAML-based configuration management
- Modular encoder/decoder architecture
- A2C training and evaluation
- XAI-ready components

## Setup: Import Framework

In [ ]:
import sys
from pathlib import Path

# Add framework to path
framework_path = Path.cwd()
sys.path.insert(0, str(framework_path))

# Import RL4EVRP framework
import rl4evrp as rl

print("✓ RL4EVRP framework imported successfully")

## Load Configuration

In [ ]:
# Initialize framework with configs
framework = rl.RL4EVRP()

# Read configurations from YAML files
problem_config = framework.read_yaml('problem')
model_config = framework.read_yaml('model')
env_config = framework.read_yaml('env')

print("Problem Configuration:")
framework.config.print_config()

## Build Model

In [ ]:
# Build model using builder pattern
model_builder = framework.build()

# Create A2C agent (contains encoder + decoder)
model = model_builder.complete_model()

print(f"Model device: {model.device}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

## Generate Training & Evaluation Data

In [ ]:
import numpy as np

# Generate training and evaluation instances
n_train = problem_config['instance_generation']['n_train_instances']
n_eval = problem_config['instance_generation']['n_eval_instances']

train_instances = [
    framework.generate_instance(seed=i)
    for i in range(n_train)
]

eval_instances = [
    framework.generate_instance(seed=1000 + i)
    for i in range(n_eval)
]

print(f"✓ Generated {len(train_instances)} training instances")
print(f"✓ Generated {len(eval_instances)} evaluation instances")

# Example instance
demo_inst = train_instances[0]
print(f"\nExample instance:")
print(f"  Nodes: {demo_inst['n_nodes']}")
print(f"  Chargers: {(demo_inst['node_types'] == 2).sum()}")
print(f"  Customers: {(demo_inst['node_types'] == 1).sum()}")

## Train Agent

In [ ]:
from rl4evrp.utils import train_agent

# Train for first seed
seeds = framework.get_seeds()
seed = seeds[0]

print(f"Training with seed: {seed}")

# Set seed
np.random.seed(seed)
import torch
torch.manual_seed(seed)

# Reinitialize model with seed
model = model_builder.complete_model()

# Train for N episodes (use small number for demo)
n_episodes = 50  # Change to 800 for full training

training_results = train_agent(
    model,
    train_instances,
    n_episodes=n_episodes,
    device=str(framework.device),
    save_dir=framework.output_dir / 'checkpoints',
    eval_instances=eval_instances,
    save_interval=10
)

print("✓ Training completed!")

## Evaluation

In [ ]:
from rl4evrp.utils import evaluate_agent

# Evaluate on test set
eval_stats = evaluate_agent(
    model,
    eval_instances,
    device=str(framework.device),
    greedy=True,
    n_eval=20
)

print("\nEvaluation Results:")
print(f"  Mean Reward: {eval_stats['mean_reward']:.3f} ± {eval_stats['std_reward']:.3f}")
print(f"  Mean Distance: {eval_stats['mean_distance']:.3f} ± {eval_stats['std_distance']:.3f}")

## Route Visualization Example

In [ ]:
import matplotlib.pyplot as plt
from rl4evrp.utils import run_episode

# Run episode and collect route
test_inst = eval_instances[0]
reward, route, dist, info, _, _, env = run_episode(
    model,
    test_inst,
    device=str(framework.device),
    greedy=True
)

# Plot route
fig, ax = plt.subplots(1, 1, figsize=(8, 8))

# Plot coordinates
coords = test_inst['coords']
types = test_inst['node_types']

# Depot
ax.plot(coords[0, 0], coords[0, 1], 'r*', markersize=20, label='Depot')

# Customers
customer_mask = types == 1
ax.scatter(coords[customer_mask, 0], coords[customer_mask, 1], c='blue', s=50, label='Customers')

# Chargers
charger_mask = types == 2
if charger_mask.any():
    ax.scatter(coords[charger_mask, 0], coords[charger_mask, 1], c='green', s=100, marker='s', label='Chargers')

# Route
route_coords = coords[route]
ax.plot(route_coords[:, 0], route_coords[:, 1], 'k-', alpha=0.3, linewidth=0.5)

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_title(f'Route: Distance={dist:.2f}, Reward={reward:.2f}')
ax.legend()
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

print(f"Route: {route}")
print(f"Battery violations: {info['batt_violations']}")
print(f"Charger visits: {info['charger_visits']}")

## Training Diagnostics

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Training rewards
axes[0, 0].plot(training_results['train_rewards'])
axes[0, 0].set_xlabel('Episode')
axes[0, 0].set_ylabel('Reward')
axes[0, 0].set_title('Training Reward')
axes[0, 0].grid(True, alpha=0.3)

# Training loss
axes[0, 1].plot(training_results['losses'])
axes[0, 1].set_xlabel('Episode')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].set_title('Training Loss')
axes[0, 1].grid(True, alpha=0.3)

# Entropy
axes[1, 0].plot(training_results['entropies'])
axes[1, 0].set_xlabel('Episode')
axes[1, 0].set_ylabel('Entropy')
axes[1, 0].set_title('Action Distribution Entropy')
axes[1, 0].grid(True, alpha=0.3)

# Evaluation rewards
if training_results['eval_rewards']:
    axes[1, 1].plot(training_results['eval_rewards'])
    axes[1, 1].set_xlabel('Evaluation Step')
    axes[1, 1].set_ylabel('Reward')
    axes[1, 1].set_title('Evaluation Reward')
    axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Save Model Checkpoint

In [ ]:
import torch

# Save model
checkpoint_path = framework.output_dir / 'final_model.pt'
torch.save(model.state_dict(), checkpoint_path)

print(f"✓ Model saved to {checkpoint_path}")

# Also save training config
import json
config_save_path = framework.output_dir / 'training_config.json'
json.dump({
    'problem': problem_config,
    'model': model_config,
    'n_episodes': n_episodes,
    'seed': seed
}, open(config_save_path, 'w'), indent=2, default=str)

print(f"✓ Config saved to {config_save_path}")